# Análise exploratória dos dados

## Importando as bibliotecas

In [2094]:
import pandas as pd
import numpy as np
import ast

## Caminho para o dataset

In [2095]:
path_csv = '../data/raw/movies_dataset.csv'

## Lendo o dataset

In [2096]:
df = pd.read_csv(path_csv, sep=',')
df

,id,title,original_title,original_language,overview,tagline,budget,revenue,runtime,release_date,genres,popularity,vote_average,vote_count
0,411405,Small Crimes,Small Crimes,en,"A disgraced former cop, fresh off a six-year p...",NaN,0.0,0.0,95.0,2017-04-28,"[{'id': 18, 'name': 'Drama'}, {'id': 35, 'name...",7.219022,5.8,55.0
1,42492,Up the Sandbox,Up the Sandbox,en,"A young wife and mother, bored with day-to-day...",NaN,0.0,0.0,97.0,1972-12-21,"[{'id': 18, 'name': 'Drama'}, {'id': 35, 'name...",0.138450,7.3,2.0
2,12143,Bad Lieutenant,Bad Lieutenant,en,"While investigating a young nun's rape, a corr...",Gambler. Thief. Junkie. Killer. Cop.,1000000.0,2019469.0,96.0,1992-09-16,"[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...",6.417037,6.9,162.0
3,9976,Satan's Little Helper,Satan's Little Helper,en,A naïve young boy unknowingly becomes the pawn...,You'll laugh 'til you die,0.0,0.0,100.0,2004-01-01,"[{'id': 27, 'name': 'Horror'}, {'id': 10749, '...",2.233189,5.0,42.0
4,46761,Sitcom,Sitcom,fr,The adventures of an upper-class suburban fami...,NaN,0.0,0.0,80.0,1998-05-27,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",1.800582,6.4,27.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4997,9803,Seven Dwarfs,7 Zwerge - Männer allein im Wald,de,The Seven Dwarves live deep within a female-fr...,NaN,0.0,0.0,95.0,2004-10-28,"[{'id': 35, 'name': 'Comedy'}]",4.582736,5.2,70.0
4998,336970,True Siblings,Syskonsalt,sv,"The siblings Linus, 19-years-old, who are taki...",NaN,0.0,0.0,58.0,2000-09-13,"[{'id': 18, 'name': 'Drama'}, {'id': 10770, 'n...",2.364355,8.0,2.0
4999,336691,Paternity Leave,Paternity Leave,en,"Four years into his first stable relationship,...",NaN,0.0,0.0,90.0,2015-11-24,"[{'id': 35, 'name': 'Comedy'}, {'id': 10749, '...",0.448056,7.0,4.0
5000,37598,Butch and Sundance: The Early Days,Butch and Sundance: The Early Days,en,"A ""prequel"" of sorts to ""Butch Cassidy and the...",Before the fame when fun was the name of the g...,0.0,0.0,115.0,1979-06-15,"[{'id': 37, 'name': 'Western'}]",0.627647,4.6,8.0


## Análise exploratória

Inicialmente, pelas informações do dataframe (DF), vemos que o range de índices corresponde ao total de linhas e o DF possui 14 colunas com os tipos: `object`, `float64` e `int64`.

In [2097]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5002 entries, 0 to 5001
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5002 non-null   int64  
 1   title              5001 non-null   object 
 2   original_title     5002 non-null   object 
 3   original_language  5001 non-null   object 
 4   overview           4904 non-null   object 
 5   tagline            2246 non-null   object 
 6   budget             4802 non-null   float64
 7   revenue            4851 non-null   float64
 8   runtime            4875 non-null   object 
 9   release_date       4888 non-null   object 
 10  genres             5002 non-null   object 
 11  popularity         5001 non-null   float64
 12  vote_average       5001 non-null   float64
 13  vote_count         5001 non-null   float64
dtypes: float64(5), int64(1), object(8)
memory usage: 547.2+ KB


Notamos que a coluna `genres` possuem dados que são strings de uma lista de dicionários, logo iremos primeiramente tratar essa coluna. 

In [2098]:
df['genres'][0]

"[{'id': 18, 'name': 'Drama'}, {'id': 35, 'name': 'Comedy'}, {'id': 53, 'name': 'Thriller'}, {'id': 80, 'name': 'Crime'}]"

Vamos transformar os dados em `genres` para uma lista de dicionários e repetir as linhas para cada item da lista.

In [2099]:
df['genres'] = df['genres'].apply(ast.literal_eval)

In [2100]:
df_exploded = df.explode('genres')

Agora, transformar os dicionários em colunas.

In [2101]:
genres_normalizados = pd.json_normalize(df_exploded['genres'])

E finalmente, juntar essas colunas ao DF original (sem a coluna antiga `genres`). 

In [2102]:
df_extendido = df.drop(columns='genres').reset_index(drop=True)
df_extendido = pd.concat([df_extendido, genres_normalizados], axis=1)

Percebemos que agora o DF posuem mais uma coluna de `id` e uma coluna `name` com os gêneros de filmes. Primeiro, vamos retirar as colunas de `id`, pois não agregam no DF.

In [2103]:
df_extendido

,id,title,original_title,original_language,overview,tagline,budget,revenue,runtime,release_date,popularity,vote_average,vote_count,id,name
0,411405.0,Small Crimes,Small Crimes,en,"A disgraced former cop, fresh off a six-year p...",NaN,0.0,0.0,95.0,2017-04-28,7.219022,5.8,55.0,18.0,Drama
1,42492.0,Up the Sandbox,Up the Sandbox,en,"A young wife and mother, bored with day-to-day...",NaN,0.0,0.0,97.0,1972-12-21,0.138450,7.3,2.0,35.0,Comedy
2,12143.0,Bad Lieutenant,Bad Lieutenant,en,"While investigating a young nun's rape, a corr...",Gambler. Thief. Junkie. Killer. Cop.,1000000.0,2019469.0,96.0,1992-09-16,6.417037,6.9,162.0,53.0,Thriller
3,9976.0,Satan's Little Helper,Satan's Little Helper,en,A naïve young boy unknowingly becomes the pawn...,You'll laugh 'til you die,0.0,0.0,100.0,2004-01-01,2.233189,5.0,42.0,80.0,Crime
4,46761.0,Sitcom,Sitcom,fr,The adventures of an upper-class suburban fami...,NaN,0.0,0.0,80.0,1998-05-27,1.800582,6.4,27.0,18.0,Drama
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10291,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10770.0,TV Movie
10292,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.0,Comedy
10293,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10749.0,Romance
10294,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.0,Western


In [2104]:
df_extendido.drop(columns=['id'], inplace=True)
df_extendido

,title,original_title,original_language,overview,tagline,budget,revenue,runtime,release_date,popularity,vote_average,vote_count,name
0,Small Crimes,Small Crimes,en,"A disgraced former cop, fresh off a six-year p...",NaN,0.0,0.0,95.0,2017-04-28,7.219022,5.8,55.0,Drama
1,Up the Sandbox,Up the Sandbox,en,"A young wife and mother, bored with day-to-day...",NaN,0.0,0.0,97.0,1972-12-21,0.138450,7.3,2.0,Comedy
2,Bad Lieutenant,Bad Lieutenant,en,"While investigating a young nun's rape, a corr...",Gambler. Thief. Junkie. Killer. Cop.,1000000.0,2019469.0,96.0,1992-09-16,6.417037,6.9,162.0,Thriller
3,Satan's Little Helper,Satan's Little Helper,en,A naïve young boy unknowingly becomes the pawn...,You'll laugh 'til you die,0.0,0.0,100.0,2004-01-01,2.233189,5.0,42.0,Crime
4,Sitcom,Sitcom,fr,The adventures of an upper-class suburban fami...,NaN,0.0,0.0,80.0,1998-05-27,1.800582,6.4,27.0,Drama
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10291,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TV Movie
10292,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Comedy
10293,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Romance
10294,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Western


## Limpando dados duplicados

Agora vamos retirar os dados duplicados.

In [2105]:
df_extendido.drop_duplicates(inplace=True)
df_extendido

,title,original_title,original_language,overview,tagline,budget,revenue,runtime,release_date,popularity,vote_average,vote_count,name
0,Small Crimes,Small Crimes,en,"A disgraced former cop, fresh off a six-year p...",NaN,0.0,0.0,95.0,2017-04-28,7.219022,5.8,55.0,Drama
1,Up the Sandbox,Up the Sandbox,en,"A young wife and mother, bored with day-to-day...",NaN,0.0,0.0,97.0,1972-12-21,0.138450,7.3,2.0,Comedy
2,Bad Lieutenant,Bad Lieutenant,en,"While investigating a young nun's rape, a corr...",Gambler. Thief. Junkie. Killer. Cop.,1000000.0,2019469.0,96.0,1992-09-16,6.417037,6.9,162.0,Thriller
3,Satan's Little Helper,Satan's Little Helper,en,A naïve young boy unknowingly becomes the pawn...,You'll laugh 'til you die,0.0,0.0,100.0,2004-01-01,2.233189,5.0,42.0,Crime
4,Sitcom,Sitcom,fr,The adventures of an upper-class suburban fami...,NaN,0.0,0.0,80.0,1998-05-27,1.800582,6.4,27.0,Drama
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5037,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Crime
5047,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Foreign
5052,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Animation
5060,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fantasy


Renomeamos a coluna `name` para `genres`.

In [2106]:
df_extendido = df_extendido.rename(columns={'name': 'genres'})
df_extendido

,title,original_title,original_language,overview,tagline,budget,revenue,runtime,release_date,popularity,vote_average,vote_count,genres
0,Small Crimes,Small Crimes,en,"A disgraced former cop, fresh off a six-year p...",NaN,0.0,0.0,95.0,2017-04-28,7.219022,5.8,55.0,Drama
1,Up the Sandbox,Up the Sandbox,en,"A young wife and mother, bored with day-to-day...",NaN,0.0,0.0,97.0,1972-12-21,0.138450,7.3,2.0,Comedy
2,Bad Lieutenant,Bad Lieutenant,en,"While investigating a young nun's rape, a corr...",Gambler. Thief. Junkie. Killer. Cop.,1000000.0,2019469.0,96.0,1992-09-16,6.417037,6.9,162.0,Thriller
3,Satan's Little Helper,Satan's Little Helper,en,A naïve young boy unknowingly becomes the pawn...,You'll laugh 'til you die,0.0,0.0,100.0,2004-01-01,2.233189,5.0,42.0,Crime
4,Sitcom,Sitcom,fr,The adventures of an upper-class suburban fami...,NaN,0.0,0.0,80.0,1998-05-27,1.800582,6.4,27.0,Drama
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5037,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Crime
5047,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Foreign
5052,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Animation
5060,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fantasy


# Tratando dados ausentes e mudando tipo de colunas

Retirando a coluna `genres` afim de excluir todos o registros que tem NaN em todas as colunas exeto em `genres`.

In [2107]:
df_sem_execao = df_extendido.drop(columns='genres')
df_sem_execao

,title,original_title,original_language,overview,tagline,budget,revenue,runtime,release_date,popularity,vote_average,vote_count
0,Small Crimes,Small Crimes,en,"A disgraced former cop, fresh off a six-year p...",NaN,0.0,0.0,95.0,2017-04-28,7.219022,5.8,55.0
1,Up the Sandbox,Up the Sandbox,en,"A young wife and mother, bored with day-to-day...",NaN,0.0,0.0,97.0,1972-12-21,0.138450,7.3,2.0
2,Bad Lieutenant,Bad Lieutenant,en,"While investigating a young nun's rape, a corr...",Gambler. Thief. Junkie. Killer. Cop.,1000000.0,2019469.0,96.0,1992-09-16,6.417037,6.9,162.0
3,Satan's Little Helper,Satan's Little Helper,en,A naïve young boy unknowingly becomes the pawn...,You'll laugh 'til you die,0.0,0.0,100.0,2004-01-01,2.233189,5.0,42.0
4,Sitcom,Sitcom,fr,The adventures of an upper-class suburban fami...,NaN,0.0,0.0,80.0,1998-05-27,1.800582,6.4,27.0
...,...,...,...,...,...,...,...,...,...,...,...,...
5037,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5047,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5052,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5060,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Excluindo todas as linhas que possuem NaN em todas as colunas exeto na coluna `genres`. Essas linhas não agregam no DF e não conseguimos inferir seus valores.

In [2108]:
df_extendido_filtro1 = df_extendido[~df_sem_execao.isna().all(axis=1)].reset_index(drop=True)
df_extendido_filtro1

,title,original_title,original_language,overview,tagline,budget,revenue,runtime,release_date,popularity,vote_average,vote_count,genres
0,Small Crimes,Small Crimes,en,"A disgraced former cop, fresh off a six-year p...",NaN,0.0,0.0,95.0,2017-04-28,7.219022,5.8,55.0,Drama
1,Up the Sandbox,Up the Sandbox,en,"A young wife and mother, bored with day-to-day...",NaN,0.0,0.0,97.0,1972-12-21,0.138450,7.3,2.0,Comedy
2,Bad Lieutenant,Bad Lieutenant,en,"While investigating a young nun's rape, a corr...",Gambler. Thief. Junkie. Killer. Cop.,1000000.0,2019469.0,96.0,1992-09-16,6.417037,6.9,162.0,Thriller
3,Satan's Little Helper,Satan's Little Helper,en,A naïve young boy unknowingly becomes the pawn...,You'll laugh 'til you die,0.0,0.0,100.0,2004-01-01,2.233189,5.0,42.0,Crime
4,Sitcom,Sitcom,fr,The adventures of an upper-class suburban fami...,NaN,0.0,0.0,80.0,1998-05-27,1.800582,6.4,27.0,Drama
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4997,Seven Dwarfs,7 Zwerge - Männer allein im Wald,de,The Seven Dwarves live deep within a female-fr...,NaN,0.0,0.0,95.0,2004-10-28,4.582736,5.2,70.0,Family
4998,True Siblings,Syskonsalt,sv,"The siblings Linus, 19-years-old, who are taki...",NaN,0.0,0.0,58.0,2000-09-13,2.364355,8.0,2.0,Fantasy
4999,Paternity Leave,Paternity Leave,en,"Four years into his first stable relationship,...",NaN,0.0,0.0,90.0,2015-11-24,0.448056,7.0,4.0,Action
5000,Butch and Sundance: The Early Days,Butch and Sundance: The Early Days,en,"A ""prequel"" of sorts to ""Butch Cassidy and the...",Before the fame when fun was the name of the g...,0.0,0.0,115.0,1979-06-15,0.627647,4.6,8.0,Animation


Verificando os dados ausentes (`NaN`) em cada coluna e tirando a média percebemos que mais de 50% dos dados na coluna `tagline` possuem valores `NaN`. A colouna `tagline` é apenas uma frase promocional curta para divulgar o filme e como é basicamente específica para cada dado podemos retirar essa coluna, pois ela não tem nenhuma ligação com outras colunas, são únicas e não conseguimos atribuir valor para os `NaN`s que são a maior parte dos dados e, assim, não agrega no DF. 

In [2109]:
df_extendido_filtro1.isnull().sum()

title                   1
original_title          0
original_language       1
overview               98
tagline              2756
budget                200
revenue               151
runtime               127
release_date          114
popularity              1
vote_average            1
vote_count              1
genres                120
dtype: int64

In [2110]:
df_extendido_filtro1.isnull().mean()

title                0.000200
original_title       0.000000
original_language    0.000200
overview             0.019592
tagline              0.550980
budget               0.039984
revenue              0.030188
runtime              0.025390
release_date         0.022791
popularity           0.000200
vote_average         0.000200
vote_count           0.000200
genres               0.023990
dtype: float64

Dropar a coluna `tagline` pois não influencia no DF e tem muito valor ausente (mais de 50%)

In [2111]:
df_extendido_filtro2 = df_extendido_filtro1.drop('tagline', axis=1)
df_extendido_filtro2

,title,original_title,original_language,overview,budget,revenue,runtime,release_date,popularity,vote_average,vote_count,genres
0,Small Crimes,Small Crimes,en,"A disgraced former cop, fresh off a six-year p...",0.0,0.0,95.0,2017-04-28,7.219022,5.8,55.0,Drama
1,Up the Sandbox,Up the Sandbox,en,"A young wife and mother, bored with day-to-day...",0.0,0.0,97.0,1972-12-21,0.138450,7.3,2.0,Comedy
2,Bad Lieutenant,Bad Lieutenant,en,"While investigating a young nun's rape, a corr...",1000000.0,2019469.0,96.0,1992-09-16,6.417037,6.9,162.0,Thriller
3,Satan's Little Helper,Satan's Little Helper,en,A naïve young boy unknowingly becomes the pawn...,0.0,0.0,100.0,2004-01-01,2.233189,5.0,42.0,Crime
4,Sitcom,Sitcom,fr,The adventures of an upper-class suburban fami...,0.0,0.0,80.0,1998-05-27,1.800582,6.4,27.0,Drama
...,...,...,...,...,...,...,...,...,...,...,...,...
4997,Seven Dwarfs,7 Zwerge - Männer allein im Wald,de,The Seven Dwarves live deep within a female-fr...,0.0,0.0,95.0,2004-10-28,4.582736,5.2,70.0,Family
4998,True Siblings,Syskonsalt,sv,"The siblings Linus, 19-years-old, who are taki...",0.0,0.0,58.0,2000-09-13,2.364355,8.0,2.0,Fantasy
4999,Paternity Leave,Paternity Leave,en,"Four years into his first stable relationship,...",0.0,0.0,90.0,2015-11-24,0.448056,7.0,4.0,Action
5000,Butch and Sundance: The Early Days,Butch and Sundance: The Early Days,en,"A ""prequel"" of sorts to ""Butch Cassidy and the...",0.0,0.0,115.0,1979-06-15,0.627647,4.6,8.0,Animation


Analisando as colunas `budget` e `revenue` percebemos que mais de 70% dos registros possuem valores 0.0 que são `NaN`s disfaçados. Logo, vamos substituir todos ele por `NaN`. 

In [2112]:
df_extendido_filtro2[df_extendido_filtro2['revenue'] == 0.0].shape[0]

4053

In [2113]:
df_extendido_filtro2[df_extendido_filtro2['budget'] == 0.0].shape[0]

3902

Substituindo `zeros` por `NaN`s nas colunas `budget` e `revenue`.

In [2114]:
df_extendido_filtro2[['budget', 'revenue']] = df_extendido_filtro2[['budget', 'revenue']].replace(0.0, np.nan)

In [2115]:
df_extendido_filtro2

,title,original_title,original_language,overview,budget,revenue,runtime,release_date,popularity,vote_average,vote_count,genres
0,Small Crimes,Small Crimes,en,"A disgraced former cop, fresh off a six-year p...",NaN,NaN,95.0,2017-04-28,7.219022,5.8,55.0,Drama
1,Up the Sandbox,Up the Sandbox,en,"A young wife and mother, bored with day-to-day...",NaN,NaN,97.0,1972-12-21,0.138450,7.3,2.0,Comedy
2,Bad Lieutenant,Bad Lieutenant,en,"While investigating a young nun's rape, a corr...",1000000.0,2019469.0,96.0,1992-09-16,6.417037,6.9,162.0,Thriller
3,Satan's Little Helper,Satan's Little Helper,en,A naïve young boy unknowingly becomes the pawn...,NaN,NaN,100.0,2004-01-01,2.233189,5.0,42.0,Crime
4,Sitcom,Sitcom,fr,The adventures of an upper-class suburban fami...,NaN,NaN,80.0,1998-05-27,1.800582,6.4,27.0,Drama
...,...,...,...,...,...,...,...,...,...,...,...,...
4997,Seven Dwarfs,7 Zwerge - Männer allein im Wald,de,The Seven Dwarves live deep within a female-fr...,NaN,NaN,95.0,2004-10-28,4.582736,5.2,70.0,Family
4998,True Siblings,Syskonsalt,sv,"The siblings Linus, 19-years-old, who are taki...",NaN,NaN,58.0,2000-09-13,2.364355,8.0,2.0,Fantasy
4999,Paternity Leave,Paternity Leave,en,"Four years into his first stable relationship,...",NaN,NaN,90.0,2015-11-24,0.448056,7.0,4.0,Action
5000,Butch and Sundance: The Early Days,Butch and Sundance: The Early Days,en,"A ""prequel"" of sorts to ""Butch Cassidy and the...",NaN,NaN,115.0,1979-06-15,0.627647,4.6,8.0,Animation


Dados de colunas como `budget` e `revenue` costumam ter distribuição assimétrica com muitos filmes com orçamento baixo e poucos com orçamento alto e muitos outliers, logo por conta da `mediana` ser mais robusta aos `outliers`, ela oferece uma estimativa mais estável e confiável. Assim, usaremos a mediana para substiruir os valores `NaN` das duas colunas.

In [2116]:
df_extendido_filtro2[df_extendido_filtro2['revenue'] > 0]['revenue'].describe()

count    7.970000e+02
mean     6.078724e+07
std      1.233708e+08
min      1.000000e+00
25%      1.929168e+06
50%      1.594263e+07
75%      6.378208e+07
max      1.156731e+09
Name: revenue, dtype: float64

In [2117]:
df_extendido_filtro2[df_extendido_filtro2['budget'] > 0]['budget'].describe()

count    9.000000e+02
mean     1.316451e+08
std      3.332803e+09
min      1.000000e+00
25%      2.000000e+06
50%      8.000000e+06
75%      2.500000e+07
max      1.000000e+11
Name: budget, dtype: float64

Faz sentido calcular a mediana por gênero tendo em vista que diferentes gêneros de filmes possuem diferentes orçamentos e receitas e substituir nos valores `NaN` do DF afim de diminuir o viés do modelo de `Machine Learning` que tenderia seria maior com a mediana geral. 

In [2118]:
median_budget = df_extendido_filtro2.groupby('genres')['budget'].median()
median_budget

genres
Action              9750000.0
Adventure           9500000.0
Animation           8000000.0
Comedy              8250000.0
Crime               9500000.0
Documentary        12000000.0
Drama               6000000.0
Family              5000000.0
Fantasy            10500000.0
Foreign            25000000.0
History             4650000.0
Horror             11500000.0
Music               5000000.0
Mystery             6800000.0
Romance             8000000.0
Science Fiction    15500000.0
TV Movie            4900000.0
Thriller           12000000.0
War                 4300000.0
Western            10000000.0
Name: budget, dtype: float64

In [2119]:
median_revenue = df_extendido_filtro2.groupby('genres')['revenue'].median()
median_revenue

genres
Action             30039392.0
Adventure          44348894.0
Animation          15185231.5
Comedy             10154165.5
Crime              15942628.0
Documentary        11341016.0
Drama              11818417.0
Family             23371723.0
Fantasy            14051384.0
Foreign            24469050.0
History            13583254.5
Horror             16641213.5
Music              10959475.0
Mystery            13318136.0
Romance            21126225.0
Science Fiction    22773535.0
TV Movie           27429860.5
Thriller           14393902.0
War                18797515.5
Western            25224242.0
Name: revenue, dtype: float64

Definimos uma função que preencherá com a mediana por gênero em cada registro.

In [2120]:
def fill_by_genre(row, column, median_dict):
    if pd.isna(row[column]):
        return median_dict.get(row['genres'], np.nan)
    return row[column]

Aqui estamos de fato preenchendo os valores ausentes com essas medianas.

In [2121]:
df_extendido_filtro2['budget'] = df_extendido_filtro2.apply(lambda row: fill_by_genre(row, 'budget', median_budget), axis=1)
df_extendido_filtro2['revenue'] = df_extendido_filtro2.apply(lambda row: fill_by_genre(row, 'revenue', median_revenue), axis=1)

In [2122]:
df_extendido_filtro2

,title,original_title,original_language,overview,budget,revenue,runtime,release_date,popularity,vote_average,vote_count,genres
0,Small Crimes,Small Crimes,en,"A disgraced former cop, fresh off a six-year p...",6000000.0,11818417.0,95.0,2017-04-28,7.219022,5.8,55.0,Drama
1,Up the Sandbox,Up the Sandbox,en,"A young wife and mother, bored with day-to-day...",8250000.0,10154165.5,97.0,1972-12-21,0.138450,7.3,2.0,Comedy
2,Bad Lieutenant,Bad Lieutenant,en,"While investigating a young nun's rape, a corr...",1000000.0,2019469.0,96.0,1992-09-16,6.417037,6.9,162.0,Thriller
3,Satan's Little Helper,Satan's Little Helper,en,A naïve young boy unknowingly becomes the pawn...,9500000.0,15942628.0,100.0,2004-01-01,2.233189,5.0,42.0,Crime
4,Sitcom,Sitcom,fr,The adventures of an upper-class suburban fami...,6000000.0,11818417.0,80.0,1998-05-27,1.800582,6.4,27.0,Drama
...,...,...,...,...,...,...,...,...,...,...,...,...
4997,Seven Dwarfs,7 Zwerge - Männer allein im Wald,de,The Seven Dwarves live deep within a female-fr...,5000000.0,23371723.0,95.0,2004-10-28,4.582736,5.2,70.0,Family
4998,True Siblings,Syskonsalt,sv,"The siblings Linus, 19-years-old, who are taki...",10500000.0,14051384.0,58.0,2000-09-13,2.364355,8.0,2.0,Fantasy
4999,Paternity Leave,Paternity Leave,en,"Four years into his first stable relationship,...",9750000.0,30039392.0,90.0,2015-11-24,0.448056,7.0,4.0,Action
5000,Butch and Sundance: The Early Days,Butch and Sundance: The Early Days,en,"A ""prequel"" of sorts to ""Butch Cassidy and the...",8000000.0,15185231.5,115.0,1979-06-15,0.627647,4.6,8.0,Animation


Algumas linhas na coluna `runtime` aparecem com um termo `min` de minutos após o número, o que  deixa essa coluna com tipo de `object` com os elementos sendo `strings`. Nessa coluna também faz sentido substituir os valores ausentes pela mediana por gênero e para isso vamos ter que primeiro mudar o tipo da coluna para `float` retirando o `min` de cada elemento que o contenha.

Retirando `min`.

In [2123]:
df_extendido_filtro2['runtime'] = df_extendido_filtro2['runtime'].str.replace('min', '', regex=False).str.strip()

Mudando o tipo da coluna para `float`.

In [2124]:
df_extendido_filtro2['runtime'] = df_extendido_filtro2['runtime'].astype(float)

In [2125]:
df_extendido_filtro2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5002 entries, 0 to 5001
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              5001 non-null   object 
 1   original_title     5002 non-null   object 
 2   original_language  5001 non-null   object 
 3   overview           4904 non-null   object 
 4   budget             4904 non-null   float64
 5   revenue            4903 non-null   float64
 6   runtime            4875 non-null   float64
 7   release_date       4888 non-null   object 
 8   popularity         5001 non-null   float64
 9   vote_average       5001 non-null   float64
 10  vote_count         5001 non-null   float64
 11  genres             4882 non-null   object 
dtypes: float64(6), object(6)
memory usage: 469.1+ KB


In [2126]:
df_extendido_filtro2.isnull().sum()

title                  1
original_title         0
original_language      1
overview              98
budget                98
revenue               99
runtime              127
release_date         114
popularity             1
vote_average           1
vote_count             1
genres               120
dtype: int64

In [2127]:
median_runtime = df_extendido_filtro2.groupby('genres')['runtime'].median()
median_runtime

genres
Action              94.5
Adventure           96.0
Animation           96.0
Comedy              95.0
Crime               95.0
Documentary         93.0
Drama               94.0
Family              92.0
Fantasy             95.5
Foreign            100.0
History             93.0
Horror              93.5
Music               93.0
Mystery             95.0
Romance             93.0
Science Fiction     94.5
TV Movie            92.0
Thriller            95.0
War                 95.0
Western             92.0
Name: runtime, dtype: float64

In [2128]:
df_extendido_filtro2['runtime'] = df_extendido_filtro2.apply(lambda row: fill_by_genre(row, 'runtime', median_revenue), axis=1)

Para as colunas `genres` e `release_date` usaremos a `moda` para preencher os valores ausentes. 

In [2129]:
mode_1 = df_extendido_filtro2['genres'].mode()[0]
df_extendido_filtro2['genres'] = df_extendido_filtro2['genres'].fillna(mode_1)

In [2130]:
mode_2 = df_extendido_filtro2['release_date'].mode()[0]
df_extendido_filtro2['release_date'] = df_extendido_filtro2['release_date'].fillna(mode_2)

In [2131]:
df_extendido_filtro2.isnull().sum()

title                 1
original_title        0
original_language     1
overview             98
budget               98
revenue              99
runtime               4
release_date          0
popularity            1
vote_average          1
vote_count            1
genres                0
dtype: int64

Como `Title` e `original_language` tem apenas um registro ausente e os valores na coluna `overview` são únicos e com poucos `NaN`s, apenas retiraremos essas linhas do DF. Além disso, faremos o mesmo com as demais colunas que ainda possuem `NaN` e no final teremos apenas eliminado pouco mais de 100 registros do DF, o que não afetará no modelo de `Machine Learning`. 

In [2132]:
df_extendido_filtro2 = df_extendido_filtro2.dropna(subset=['overview'])
df_extendido_filtro2 = df_extendido_filtro2.dropna(subset=['revenue'])
df_extendido_filtro2 = df_extendido_filtro2.dropna(subset=['budget'])
df_extendido_filtro2 = df_extendido_filtro2.dropna(subset=['runtime'])
df_extendido_filtro2 = df_extendido_filtro2.dropna(subset=['title'])
df_extendido_filtro2 = df_extendido_filtro2.dropna(subset=['original_language'])



In [2133]:
df_extendido_filtro2.isnull().sum()

title                0
original_title       0
original_language    0
overview             0
budget               0
revenue              0
runtime              0
release_date         0
popularity           0
vote_average         0
vote_count           0
genres               0
dtype: int64

Por fim, vamos mudar o tipo da coluna `release_date` para `datetime64[ns]`.

In [2134]:
df_extendido_filtro2['release_date'] = pd.to_datetime(df_extendido_filtro2['release_date'], format= 'mixed', dayfirst=True)

In [2135]:
df_extendido_filtro2['release_date'][0]

Timestamp('2017-04-28 00:00:00')

Agora todas as colunas estão com os tipo corretos.

In [2136]:
df_filmes_limpos = df_extendido_filtro2.reset_index(drop=True)
df_filmes_limpos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4793 entries, 0 to 4792
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   title              4793 non-null   object        
 1   original_title     4793 non-null   object        
 2   original_language  4793 non-null   object        
 3   overview           4793 non-null   object        
 4   budget             4793 non-null   float64       
 5   revenue            4793 non-null   float64       
 6   runtime            4793 non-null   float64       
 7   release_date       4793 non-null   datetime64[ns]
 8   popularity         4793 non-null   float64       
 9   vote_average       4793 non-null   float64       
 10  vote_count         4793 non-null   float64       
 11  genres             4793 non-null   object        
dtypes: datetime64[ns](1), float64(6), object(5)
memory usage: 449.5+ KB


## Salvando o dataframe 

In [2137]:
df_filmes_limpos.to_csv('../data/processed/filmes_limpos.csv', index=False)